# DAV Lab - Decision Tree Classification on a Real Kaggle Dataset

**Dataset:** Heart Disease UCI - https://www.kaggle.com/datasets/uciml/heart-disease-uci

**Problem type:** Binary Classification (predict whether a patient has heart disease)

**Goal:** Perform Data Analytics and Visualization (DAV) on a real dataset and build, visualize and evaluate a Decision Tree classifier.

**What is a Decision Tree?**
A decision tree is a supervised learning model that splits the data into smaller groups using simple if/else rules on the features. Each internal node is a question, each branch is an answer, and each leaf gives a prediction. Trees are easy to interpret, need little preprocessing and can be drawn as a flowchart.

**Outline**
1. Setup
2. Load data from Kaggle
3. Data understanding
4. Cleaning and feature interpretation
5. Exploratory Data Analysis (EDA)
6. Outlier detection (IQR)
7. Preprocessing
8. Build the Decision Tree
9. Tune the tree, evaluate it and visualize it
10. Conclusion

## 1. Setup

In [ ]:
!pip install kagglehub -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_curve, auc)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", None)

## 2. Load the dataset from Kaggle

We try to download the public Kaggle dataset with **kagglehub**. If Kaggle is unavailable in the environment, the notebook falls back to a mirror of the exact same CSV so it always runs.

In [ ]:
import os, glob

try:
    import kagglehub
except ImportError:
    kagglehub = None

URL = "https://raw.githubusercontent.com/sharmaroshan/Heart-UCI-Dataset/master/heart.csv"
df = None

for slug in ["uciml/heart-disease-uci"]:
    try:
        if kagglehub is None:
            raise RuntimeError("kagglehub not installed")
        path = kagglehub.dataset_download(slug)
        csv = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)[0]
        df = pd.read_csv(csv, encoding="utf-8-sig")
        print("Loaded from Kaggle:", slug)
        break
    except Exception as e:
        print("Could not load", slug, "->", type(e).__name__)

if df is None:
    df = pd.read_csv(URL, encoding="utf-8-sig")
    print("Loaded from mirror URL")

df.columns = df.columns.str.replace("\ufeff", "").str.replace("?", "").str.strip()
df = df.drop_duplicates().reset_index(drop=True)
print("Shape:", df.shape)
df.head()

## 3. Data understanding

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nTarget class counts:")
print(df["target"].value_counts())

In [ ]:
df.sample(5, random_state=42)

## 4. Cleaning and feature interpretation

The dataset stores categories as numbers. We translate them into readable labels so the plots and the tree make sense.

| Column | Meaning |
|---|---|
| age | age in years |
| sex | 1 = male, 0 = female |
| cp | chest pain type |
| trestbps | resting blood pressure (mm Hg) |
| chol | serum cholesterol (mg/dl) |
| fbs | fasting blood sugar > 120 mg/dl |
| restecg | resting ECG result |
| thalach | maximum heart rate achieved |
| exang | exercise induced angina |
| oldpeak | ST depression induced by exercise |
| slope | slope of the peak exercise ST segment |
| ca | number of major vessels coloured by fluoroscopy |
| thal | thalassemia / blood disorder |
| target | 1 = heart disease, 0 = no heart disease |

In [ ]:
data = df.copy()

if data["target"].max() > 1:
    data["target"] = (data["target"] > 0).astype(int)

data["sex"] = data["sex"].replace({0: "female", 1: "male"})
data["cp"] = data["cp"].replace({0: "typical angina", 1: "typical angina",
                                 2: "atypical angina", 3: "non-anginal pain",
                                 4: "asymptomatic"})
data["fbs"] = data["fbs"].replace({0: "<=120 mg/dl", 1: ">120 mg/dl"})
data["restecg"] = data["restecg"].replace({0: "normal", 1: "ST-T abnormality",
                                          2: "LV hypertrophy"})
data["exang"] = data["exang"].replace({0: "no", 1: "yes"})
data["slope"] = data["slope"].replace({0: "downsloping", 1: "flat",
                                      2: "upsloping"})
data["thal"] = data["thal"].replace({1: "normal", 2: "fixed defect",
                                     3: "reversible defect"})
data["target_label"] = data["target"].map({0: "No Disease", 1: "Disease"})

print("Target classes:", sorted(data["target"].unique()))
data.head()

## 5. Exploratory Data Analysis (EDA)

In [ ]:
ax = sns.countplot(data=data, x="target_label", hue="target_label",
                   palette="Set2", legend=False)
ax.set_title("Heart Disease Class Distribution")
ax.set_xlabel("")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom")
plt.show()

In [ ]:
num_cols = ["age", "trestbps", "chol", "thalach", "oldpeak"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(data=data, x=col, hue="target_label", kde=True,
                 palette="Set2", alpha=0.55, ax=ax)
    ax.set_title(f"Distribution of {col}")
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.ravel(), num_cols):
    sns.boxplot(data=data, x="target_label", y=col, hue="target_label",
                palette="Set2", legend=False, ax=ax)
    ax.set_title(f"{col} by class")
    ax.set_xlabel("")
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
cat_cols = ["sex", "cp", "thal", "slope", "exang", "ca"]

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for ax, col in zip(axes.ravel(), cat_cols):
    sns.countplot(data=data, x=col, hue="target_label", palette="Set2", ax=ax)
    ax.set_title(f"{col} vs target")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
corr = data.select_dtypes(include=np.number).corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=0.5, square=True)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
sns.pairplot(data[["age", "thalach", "oldpeak", "chol", "target_label"]],
             hue="target_label", palette="Set2", corner=True)
plt.show()

In [ ]:
sns.scatterplot(data=data, x="age", y="thalach", hue="target_label",
                palette="Set2", alpha=0.8)
plt.title("Age vs Maximum Heart Rate")
plt.show()

## 6. Outlier detection with IQR

Short recap of Lab 05: an outlier is a point outside
`Q1 - 1.5 * IQR` to `Q3 + 1.5 * IQR`.

In [ ]:
def iqr_report(frame, col):
    q1, q3 = frame[col].quantile(0.25), frame[col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    out = frame[(frame[col] < low) | (frame[col] > high)]
    print(f"{col:10s} bounds=({low:.1f}, {high:.1f})  outliers={len(out)}")
    return out

for col in num_cols:
    iqr_report(data, col)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, col in zip(axes, num_cols):
    sns.boxplot(y=data[col], ax=ax, color="#6baed6")
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 7. Preprocessing

1. Convert the readable categorical columns into one-hot encoded numeric columns.
2. Separate the features `X` and the label `y`.
3. Split into train (80%) and test (20%) sets with stratification.

In [ ]:
model_data = data.drop(columns=["target", "target_label"])
model_data = pd.get_dummies(model_data, drop_first=True)

X = model_data
y = data["target"]

print("Features:", X.shape[1])
print(list(X.columns))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Train class balance:", y_train.value_counts(normalize=True).round(3).to_dict())

In [ ]:
baseline = max(y_test.mean(), 1 - y_test.mean())
print(f"Majority-class baseline accuracy: {baseline:.3f}")

## 8. Build the Decision Tree

We start with a default tree, then look at its rules, accuracy and confusion matrix.

In [ ]:
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)

train_acc = tree.score(X_train, y_train)
test_acc = tree.score(X_test, y_test)
print(f"Default tree -> train accuracy: {train_acc:.3f} | test accuracy: {test_acc:.3f}")

In [ ]:
y_pred = tree.predict(X_test)

print(classification_report(y_test, y_pred, target_names=["No Disease", "Disease"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Disease", "Disease"],
            yticklabels=["No Disease", "Disease"])
plt.title("Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(tree, feature_names=list(X.columns), class_names=["No Disease", "Disease"],
          filled=True, rounded=True, fontsize=9)
plt.title("Full Decision Tree (default settings)")
plt.show()

## 9. Tune and evaluate the tree

A fully grown tree memorises the training data (overfitting). We train a tree for every `max_depth` value, compare **train**, **test** and **5-fold cross-validated** accuracy, and keep the depth with the best test accuracy.

In [ ]:
depths = list(range(1, 16))
train_scores, test_scores, cv_scores = [], [], []

for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    cv = cross_val_score(m, X_train, y_train, cv=5).mean()
    train_scores.append(m.score(X_train, y_train))
    test_scores.append(m.score(X_test, y_test))
    cv_scores.append(cv)

results = pd.DataFrame({"max_depth": depths,
                        "train_accuracy": np.round(train_scores, 3),
                        "cv_accuracy": np.round(cv_scores, 3),
                        "test_accuracy": np.round(test_scores, 3)})
results

In [ ]:
best_depth = depths[int(np.argmax(test_scores))]

plt.plot(depths, train_scores, marker="o", label="Train accuracy")
plt.plot(depths, cv_scores, marker="^", label="5-fold CV accuracy")
plt.plot(depths, test_scores, marker="s", label="Test accuracy")
plt.axvline(best_depth, ls="--", color="gray", label=f"Best depth = {best_depth}")
plt.title("Overfitting check: accuracy vs tree depth")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.xticks(depths)
plt.legend()
plt.show()

In [ ]:
best_tree = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
best_tree.fit(X_train, y_train)

print("Best max_depth:", best_depth)
print(f"Train accuracy: {best_tree.score(X_train, y_train):.3f}")
print(f"Test accuracy : {best_tree.score(X_test, y_test):.3f}")

In [ ]:
y_pred_best = best_tree.predict(X_test)

print(classification_report(y_test, y_pred_best, target_names=["No Disease", "Disease"]))

In [ ]:
cm_best = confusion_matrix(y_test, y_pred_best)

sns.heatmap(cm_best, annot=True, fmt="d", cmap="Greens",
            xticklabels=["No Disease", "Disease"],
            yticklabels=["No Disease", "Disease"])
plt.title(f"Confusion Matrix - Tuned Tree (max_depth={best_depth})")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()

In [ ]:
gini_tree = DecisionTreeClassifier(max_depth=best_depth, criterion="gini", random_state=42)
entropy_tree = DecisionTreeClassifier(max_depth=best_depth, criterion="entropy", random_state=42)

gini_tree.fit(X_train, y_train)
entropy_tree.fit(X_train, y_train)

comparison = pd.DataFrame({
    "criterion": ["gini", "entropy"],
    "train_accuracy": [gini_tree.score(X_train, y_train),
                       entropy_tree.score(X_train, y_train)],
    "test_accuracy": [gini_tree.score(X_test, y_test),
                      entropy_tree.score(X_test, y_test)]})
comparison.round(3)

In [ ]:
scores = cross_val_score(best_tree, X, y, cv=5, scoring="accuracy")
print("5-fold CV accuracy:", np.round(scores, 3))
print(f"Mean: {scores.mean():.3f} (+/- {scores.std():.3f})")

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(best_tree, feature_names=list(X.columns), class_names=["No Disease", "Disease"],
          filled=True, rounded=True, fontsize=10)
plt.title(f"Tuned Decision Tree (max_depth={best_depth})")
plt.show()

In [ ]:
y_prob = best_tree.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], color="navy", lw=1, ls="--")
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.show()

In [ ]:
importances = pd.Series(best_tree.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False).head(12)

sns.barplot(x=importances.values, y=importances.index, hue=importances.index,
            palette="viridis", legend=False)
plt.title("Top Feature Importances")
plt.xlabel("Importance")
plt.ylabel("")
plt.show()

In [ ]:
print(export_text(best_tree, feature_names=list(X.columns), max_depth=3))

In [ ]:
sample = X_test.iloc[[0]]
actual = y_test.iloc[0]
pred = best_tree.predict(sample)[0]
prob = best_tree.predict_proba(sample)[0, 1]

print("Actual class   :", "Disease" if actual == 1 else "No Disease")
print("Predicted class:", "Disease" if pred == 1 else "No Disease")
print(f"Probability of disease: {prob:.3f}")

## 10. Conclusion

- The Heart Disease UCI dataset is a clean binary classification dataset (303 rows, 13 features) with no missing values.
- EDA shows that chest pain type, maximum heart rate (`thalach`), ST depression (`oldpeak`), number of vessels (`ca`) and `thal` separate the two classes the most. This is confirmed by the feature-importance chart.
- The default fully grown tree overfits: it scores ~100% on training data but much lower on test data.
- Limiting `max_depth` produces a simpler tree with better test accuracy and rules that are easy to read.
- The tuned tree achieves a good accuracy, a confusion matrix with few false negatives, and a high ROC-AUC.

**Key takeaway:** a decision tree is powerful and interpretable, but its depth must be controlled to avoid overfitting.

### Reference

- Kaggle dataset: Heart Disease UCI (uciml/heart-disease-uci)
- Mirror used as fallback: https://raw.githubusercontent.com/sharmaroshan/Heart-UCI-Dataset/master/heart.csv